# 01 · Data: count, group photos of the same leaf, split

Phase 1 of the project. This notebook:

1. finds the 10 tomato classes in PlantVillage (the `color` folder) and counts the images,
2. groups photos of the same leaf (near-identical photos by an image hash, and photos named after the same leaf),
   so the same leaf is never in two splits,
3. splits the images 70 / 15 / 15 into train / val / test, the same share for every class, with a fixed seed,
4. saves `data/splits.csv` and the check figures in `results/`.

**How to run it on Kaggle**

- **Add Input** → search `plantvillage-dataset` (by *abdallahalidev*) → **Add**.
- **Settings**: Accelerator **None**. This notebook needs no GPU, so your GPU hours are saved. Internet can stay off.
- **Run All** (about 3–5 minutes) and look at the figures (see the checklist at the end).
- Then **Save Version → Save & Run All (Commit)**. Notebook 02 reads this notebook's output, so it must be saved.
- Download the output (`data/` and `results/`) into the project folder on your laptop.

Running it again with the same settings gives exactly the same split. Do not change `SEED` or `HASH_THRESHOLD`
after model training has started.

In [ ]:
import os, json, hashlib
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.fft import dct
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

SEED = 42
SPLITS = ['train', 'val', 'test']
SPLIT_FRACTIONS = {'train': 0.70, 'val': 0.15, 'test': 0.15}
HASH_THRESHOLD = 6   # two photos whose hashes differ in at most this many of the 64 bits count as the same photo
SHOW_UP_TO = 12      # pairs up to this distance are also collected, only to show them in the check figure
IMAGE_EXTS = {'.jpg', '.jpeg', '.png'}

# Kaggle folders (the two environment variables are only for a test run on another computer)
INPUT_ROOT = Path(os.environ.get('INPUT_ROOT', '/kaggle/input'))
OUT_DIR = Path(os.environ.get('OUT_DIR', '/kaggle/working'))
DATA_OUT, RESULTS_OUT = OUT_DIR / 'data', OUT_DIR / 'results'
DATA_OUT.mkdir(parents=True, exist_ok=True)
RESULTS_OUT.mkdir(parents=True, exist_ok=True)

# Chart style for all figures of the project: light background, dark text, thin grey grid, one blue
SURFACE, INK, INK_2, GRID, BLUE = '#fcfcfb', '#0b0b0b', '#52514e', '#e4e3df', '#2a78d6'
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'text.color': INK, 'axes.labelcolor': INK_2, 'xtick.color': INK_2, 'ytick.color': INK_2,
    'axes.edgecolor': GRID, 'axes.linewidth': 1, 'grid.color': GRID, 'grid.linewidth': 1,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'savefig.dpi': 150, 'savefig.bbox': 'tight',
})

## 1. Find the tomato classes and count the images

In [ ]:
CLASS_NAMES = {
    'Tomato___Bacterial_spot': 'Bacterial spot',
    'Tomato___Early_blight': 'Early blight',
    'Tomato___Late_blight': 'Late blight',
    'Tomato___Leaf_Mold': 'Leaf mold',
    'Tomato___Septoria_leaf_spot': 'Septoria leaf spot',
    'Tomato___Spider_mites Two-spotted_spider_mite': 'Spider mites',
    'Tomato___Target_Spot': 'Target spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 'Yellow leaf curl virus',
    'Tomato___Tomato_mosaic_virus': 'Mosaic virus',
    'Tomato___healthy': 'Healthy',
}

def find_color_dir(root):
    """Find PlantVillage's `color` folder (the one that holds the Tomato___ class folders) under root."""
    for dirpath, dirnames, _ in os.walk(root):
        dirnames.sort()
        if Path(dirpath).name.lower() == 'color' and any(d.startswith('Tomato') for d in dirnames):
            return Path(dirpath)
        dirnames[:] = [d for d in dirnames if '___' not in d]   # class folders only hold images: skip them
    raise FileNotFoundError(f'No PlantVillage "color" folder under {root}. '
                            'Add the Kaggle dataset abdallahalidev/plantvillage-dataset as input.')

by_name = lambda p: p.name   # plain, case-sensitive name order on every computer (Windows paths sort ignoring case)
COLOR_DIR = find_color_dir(INPUT_ROOT)
class_dirs = sorted((d for d in COLOR_DIR.iterdir() if d.is_dir() and d.name.startswith('Tomato')), key=by_name)
assert len(class_dirs) == 10, f'Expected 10 tomato classes, found {len(class_dirs)}: {[d.name for d in class_dirs]}'

# label numbers follow the sorted folder names (the same order PyTorch's ImageFolder uses)
classes = [{'label': i, 'folder': d.name,
            'name': CLASS_NAMES.get(d.name, d.name.split('___')[-1].replace('_', ' '))}
           for i, d in enumerate(class_dirs)]
df = pd.DataFrame([{'path': f'{d.name}/{f.name}', 'class_folder': d.name, 'class_name': c['name'], 'label': c['label']}
                   for c, d in zip(classes, class_dirs)
                   for f in sorted(d.iterdir(), key=by_name) if f.suffix.lower() in IMAGE_EXTS])
print('Dataset folder:', COLOR_DIR)
print(f'{len(df):,} images in {len(classes)} classes')

In [ ]:
counts = df['class_name'].value_counts()   # largest class first
print(counts.to_string())
print(f'\nThe largest class has {counts.max() / counts.min():.1f} times the images of the smallest.')

fig, ax = plt.subplots(figsize=(8, 4.8))
y = np.arange(len(counts))
ax.barh(y, counts.values, height=0.6, color=BLUE)
for yi, v in zip(y, counts.values):
    ax.text(v + counts.max() * 0.01, yi, f'{v:,}', va='center', fontsize=10, color=INK)
ax.set_yticks(y, counts.index, color=INK)
ax.invert_yaxis()
ax.tick_params(axis='y', length=0)
ax.spines['left'].set_visible(False)
ax.xaxis.grid(True)
ax.set_axisbelow(True)
ax.set_xlim(0, counts.max() * 1.12)
ax.set_xlabel('Images')
ax.set_title(f'Images per class: {len(df):,} tomato leaf photos (PlantVillage)')
fig.savefig(RESULTS_OUT / '01_class_counts.png')
plt.show()

## 2. Find copies and photos of the same leaf

PlantVillage holds several photos of the same leaf. If one is in train and another in test, the model has already
seen the test leaf and the test score is too high. So photos of the same leaf are put in a **group**, and a group
always stays in one split. Two things put photos in a group:

1. **Image hash.** Each image gets a **perceptual hash**: 64 bits that describe its coarse light-and-dark pattern. Two
   photos whose hashes differ in only a few bits look almost the same. Flipped and rotated copies are compared too.
2. **File names.** Some PlantVillage series name each photo after its leaf: `GH_HL Leaf 188`, `GH_HL Leaf 188.1` and
   `GH_HL Leaf 188.2` are the same healthy leaf photographed three times, and `GHLB Leaf 8 Day 6` and
   `GHLB Leaf 8 Day 12` are one late blight leaf on two days. Photos with the same series and leaf number go in one
   group. The hash misses most of these, because the leaf is placed and framed a little differently each time.

A photo of the same leaf taken again with no leaf number in its name, and from a clearly different angle, is caught by
neither; that is a limit of this method.

This cell also opens every image fully, so broken files are found and left out.

In [ ]:
def phash_variants(img):
    """64-bit perceptual hash of the image in each of its 8 flips/rotations, shape (8, 64)."""
    a = np.asarray(img.convert('L').resize((32, 32), Image.LANCZOS), dtype=np.float32)
    out = []
    for b in (a, np.fliplr(a)):
        for k in range(4):
            low = dct(dct(np.rot90(b, k), axis=0, norm='ortho'), axis=1, norm='ortho')[:8, :8]
            out.append((low > np.median(low)).ravel())
    return np.stack(out)

hashes, md5s, sizes, bad = [], [], [], []
ok = np.ones(len(df), dtype=bool)
for n, rel in enumerate(df['path']):
    f = COLOR_DIR / rel
    try:
        with Image.open(f) as img:
            img.load()
            h, size = phash_variants(img), img.size
    except Exception as e:
        bad.append({'path': rel, 'error': str(e)})
        ok[n] = False
        continue
    hashes.append(h)
    sizes.append(size)
    md5s.append(hashlib.md5(f.read_bytes()).hexdigest())
    if (n + 1) % 2000 == 0:
        print(f'{n + 1:,} / {len(df):,} images read')

df = df[ok].reset_index(drop=True)
df['md5'] = md5s
hashes = np.stack(hashes)   # (images, 8 flips/rotations, 64 bits)
N = len(df)
print(f'\n{N:,} images read, {len(bad)} unreadable left out {[b["path"] for b in bad]}')
print('Image sizes (width, height):', dict(Counter(sizes).most_common(5)))
print('Exact copies (same file content):', int(df['md5'].duplicated().sum()))

In [ ]:
A = hashes[:, 0].astype(np.float32)                # each image as it is
B = hashes.reshape(N * 8, 64).astype(np.float32)   # each image in all 8 flips/rotations
a_ones, b_ones = A.sum(1), B.sum(1)

nn_dist = np.empty(N, dtype=int)   # distance of each image to its most similar other image
pairs = []
for i0 in range(0, N, 128):
    i1 = min(i0 + 128, N)
    # number of different bits = bits set in a + bits set in b - 2 * bits set in both
    d = a_ones[i0:i1, None] + b_ones[None, :] - 2 * (A[i0:i1] @ B.T)
    d = d.reshape(i1 - i0, N, 8).min(axis=2).astype(int)   # the closest flip/rotation counts
    d[np.arange(i1 - i0), np.arange(i0, i1)] = 64           # an image is not a copy of itself
    nn_dist[i0:i1] = d.min(axis=1)
    r, c = np.nonzero(d <= SHOW_UP_TO)
    keep = c > r + i0                                       # each pair once
    r, c = r[keep], c[keep]
    pairs.append(np.stack([r + i0, c, d[r, c]], axis=1))
pairs = pd.DataFrame(np.concatenate(pairs), columns=['i', 'j', 'dist'])
close = pairs[pairs['dist'] <= HASH_THRESHOLD]

# photos of the same leaf by file name: '<series> Leaf <N>[.k] [Day <d>]', e.g. 'GH_HL Leaf 188.1', 'GHLB Leaf 8 Day 12'
file_name = df['path'].str.split('/').str[-1].str.split('___', n=1).str[-1]
parts = file_name.str.extract(r'^(.*?)\s*[Ll]eaf\s+(\d+)')
leaf_id = (parts[0].str.strip() + ' Leaf ' + parts[1]).dropna()    # only the names that have a leaf number
first = leaf_id.index.to_series().groupby(leaf_id).transform('first')
same_leaf = first[first != first.index]                            # each further photo -> the first photo of its leaf

def groups_of(i, j):
    """Images joined by pairs (directly or through other images) form one group."""
    graph = coo_matrix((np.ones(len(i)), (i, j)), shape=(N, N))
    return connected_components(graph, directed=False)

_, hash_group = groups_of(close['i'].to_numpy(), close['j'].to_numpy())
missed = hash_group[same_leaf.index.to_numpy()] != hash_group[same_leaf.to_numpy()]   # the hash had not joined them
n_groups, df['group'] = groups_of(np.concatenate([close['i'].to_numpy(), same_leaf.to_numpy()]),
                                  np.concatenate([close['j'].to_numpy(), same_leaf.index.to_numpy()]))
group_size = df.groupby('group')['path'].transform('size')

print(f'Hash: {len(close):,} near-identical pairs (at most {HASH_THRESHOLD} of 64 bits different)')
print(f'File names: {len(leaf_id):,} photos of {leaf_id.nunique():,} numbered leaves; {len(same_leaf):,} are a further '
      f'photo of a leaf, and the hash had missed {missed.sum():,} of those')
print(f'{N:,} images form {n_groups:,} groups; {(group_size > 1).sum():,} images share a group with another image')
size_bins = pd.cut(group_size, [0, 1, 2, 5, 10, np.inf], labels=['1 (alone)', '2', '3-5', '6-10', 'over 10'])
print('Images by the size of their group:', size_bins.value_counts(sort=False).to_dict())
print('\nImages that share a group with another image, per class:')
print(df.assign(grouped=group_size > 1).groupby('class_name')['grouped'].sum().reindex(counts.index).to_string())

classes_per_group = df.groupby('group')['label'].nunique()
mixed = classes_per_group.index[classes_per_group > 1]
print(f'\n{len(mixed)} groups hold photos from more than one class. Most are chance hash matches between different '
      'leaves, a few are\nthe same photo filed under two classes. Both only keep photos in the same split, so they are harmless.')
if len(mixed):
    print(df[df['group'].isin(mixed)].sort_values('group')[['group', 'path']].head(20).to_string(index=False))

### Check the grouping by eye

1. For every image, how many bits separate it from its most similar other image.
2. Example pairs at each distance. Most pairs in the **same photo** rows should be the same photo (maybe flipped or a
   little shifted), and the **kept apart** rows should mostly be different leaves.
3. The largest groups.
4. Photos of one leaf that only the file names joined. Each row should be one leaf (or one plant, on different days).

The hash cannot separate PlantVillage photos perfectly, because every photo is one leaf in the middle of a plain
background. So a few *same photo* pairs are different leaves, and a large group can be a chain of different leaves
(A looks like B, B looks like C). That is harmless: it only keeps unrelated photos in the same split and never puts a
copy in two splits. The other error, a copy that is not joined, is the one that matters; the file names catch many.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.hist(np.clip(nn_dist, 0, 32), bins=np.arange(34) - 0.5, color=BLUE, rwidth=0.8)
ax.axvline(HASH_THRESHOLD + 0.5, color=INK_2, linewidth=1)
ax.text(HASH_THRESHOLD + 1, ax.get_ylim()[1] * 0.92, f'threshold {HASH_THRESHOLD}: left of this line counts as a copy',
        color=INK_2, fontsize=10, va='top')
ax.yaxis.grid(True)
ax.set_axisbelow(True)
ax.set_xlabel('Bits different from the most similar other image (32 means 32 or more)')
ax.set_ylabel('Images')
ax.set_title('How close each image is to its nearest other image')
fig.savefig(RESULTS_OUT / '01_nearest_distance.png')
plt.show()

def show(ax, i, title=None):
    with Image.open(COLOR_DIR / df.at[i, 'path']) as img:
        ax.imshow(img.convert('RGB'))
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=8, loc='center', fontweight='normal', color=INK_2)

def grid_title(fig, text):
    """Title at the top left of an image grid, without the wide empty band matplotlib leaves by default."""
    h = fig.get_figheight()
    fig.subplots_adjust(top=1 - 0.6 / h)
    fig.suptitle(text, x=0.02, y=1 - 0.1 / h, ha='left', fontweight='bold')

rng = np.random.default_rng(SEED)
distances = list(range(0, SHOW_UP_TO + 1, 2))
fig = plt.figure(figsize=(12.5, 2.3 * len(distances)))
gs = fig.add_gridspec(len(distances), 8, width_ratios=[1, 1, 0.3, 1, 1, 0.3, 1, 1], wspace=0.08)  # gap between pairs
for r, dist in enumerate(distances):
    row = [fig.add_subplot(gs[r, col]) for col in (0, 1, 3, 4, 6, 7)]
    found = pairs[pairs['dist'] == dist]
    pick = found.iloc[rng.choice(len(found), min(3, len(found)), replace=False)]
    for ax in row:
        ax.axis('off')
    for k, (i, j) in enumerate(zip(pick['i'], pick['j'])):
        show(row[2 * k], i, df.at[i, 'class_name'])
        show(row[2 * k + 1], j, df.at[j, 'class_name'])
    verdict = 'same photo' if dist <= HASH_THRESHOLD else 'kept apart'
    row[0].text(-0.1, 0.5, f'distance {dist}\n{verdict}\n{len(found):,} pair{"" if len(found) == 1 else "s"}',
                transform=row[0].transAxes, ha='right', va='center', fontsize=10)
grid_title(fig, 'Example pairs at each distance (two images side by side = one pair)')
fig.savefig(RESULTS_OUT / '01_duplicate_pairs.png', dpi=100)
plt.show()

largest = df['group'].value_counts()
largest = largest[largest > 1].head(4)
if len(largest):
    fig, axes = plt.subplots(len(largest), 8, figsize=(14, 2 * len(largest)), squeeze=False)
    for row, (g, n) in zip(axes, largest.items()):
        for ax in row:
            ax.axis('off')
        for ax, i in zip(row, df.index[df['group'] == g][:8]):
            show(ax, i, df.at[i, 'class_name'])
        row[0].text(-0.1, 0.5, f'group of\n{n} images', transform=row[0].transAxes, ha='right', va='center')
    grid_title(fig, 'The largest groups (up to 8 images each)')
    fig.savefig(RESULTS_OUT / '01_largest_groups.png', dpi=100)
    plt.show()

# leaves whose photos only the file names joined (the hash had put them in different groups)
missed_leaves = leaf_id.loc[same_leaf.index[missed]].unique()
if len(missed_leaves):
    shown = rng.choice(missed_leaves, min(5, len(missed_leaves)), replace=False)
    fig, axes = plt.subplots(len(shown), 6, figsize=(11, 2.1 * len(shown)), squeeze=False)
    for row, leaf in zip(axes, shown):
        members = leaf_id.index[leaf_id == leaf]
        for ax in row:
            ax.axis('off')
        for ax, i in zip(row, members[:6]):
            show(ax, i, file_name[i].rsplit('.', 1)[0])
        row[0].text(-0.1, 0.5, f'{leaf}\n{df.at[members[0], "class_name"]}\n{len(members)} photos',
                    transform=row[0].transAxes, ha='right', va='center', fontsize=10)
    grid_title(fig, 'Photos of the same leaf, joined by file name (the hash had missed them)')
    fig.savefig(RESULTS_OUT / '01_same_leaf_by_name.png', dpi=100)
    plt.show()

## 3. Split 70 / 15 / 15

The split works on groups, not single images, so all photos of a leaf land in the same split. Within each class,
the groups are shuffled with the fixed seed and handed out one by one, each to the split that is furthest behind its
share. Because the order is random, groups of every size reach every split in about the 70 / 15 / 15 ratio, so val
and test hold the same mix of photos as train. (Handing out the biggest groups first would put all of them in train,
and val and test would then only hold photos that have no sibling photo.) A group with photos from two classes counts
under the class most of its photos have.

In [ ]:
group_label = df.groupby('group')['label'].agg(lambda s: s.mode().min())   # the class most of its photos have
group_n = df.groupby('group').size()
rng = np.random.default_rng(SEED)
split_of_group = {}
for label in range(len(classes)):
    taken = dict.fromkeys(SPLITS, 0)
    for g in rng.permutation(group_label.index[group_label == label].to_numpy()):   # random order, groups of any size
        s = min(SPLITS, key=lambda s: taken[s] / SPLIT_FRACTIONS[s])               # the split furthest behind its share
        split_of_group[g] = s
        taken[s] += group_n[g]
df['split'] = df['group'].map(split_of_group)

table = pd.crosstab(df['class_name'], df['split'], margins=True, margins_name='total')
table = table.reindex(index=[*counts.index, 'total'], columns=[*SPLITS, 'total'])
for s in SPLITS:
    table[f'{s} %'] = (100 * table[s] / table['total']).round(1)
print(table.to_string())
off = (table.loc[counts.index, [f'{s} %' for s in SPLITS]] - [100 * SPLIT_FRACTIONS[s] for s in SPLITS]).abs()
print(f'\nLargest gap from 70 / 15 / 15 in any class: {off.max().max():.1f} percentage points')
kind = pd.Series(np.where(group_size > 1, 'shares a group', 'alone'), index=df.index, name='photos')
print('\nWhere the photos that share a group went, compared with the rest (% of each row):')
print(pd.crosstab(kind, df['split'], normalize='index').mul(100).round(1)[SPLITS].to_string())

# the checks this phase is "done" by
assert df['split'].notna().all(), 'an image has no split'
assert (df.groupby('group')['split'].nunique() == 1).all(), 'a group is in two splits'
assert (df.groupby('md5')['split'].nunique() == 1).all(), 'an exact copy is in two splits'
assert (df.loc[leaf_id.index].groupby(leaf_id)['split'].nunique() == 1).all(), 'a numbered leaf is in two splits'
assert (table.loc[counts.index, SPLITS] > 0).all().all(), 'a class is missing from a split'
print('\nAll checks passed: every class is in all three splits, and no group, exact copy or numbered leaf is in two splits.')

## 4. Sample images per class

Eight random training images per class. Each row should look like its disease, with no wrong or broken images.

In [ ]:
rng = np.random.default_rng(SEED)
train = df[df['split'] == 'train']
fig, axes = plt.subplots(len(classes), 8, figsize=(16, 2.1 * len(classes)), squeeze=False)
for row, name in zip(axes, counts.index):
    members = train.index[train['class_name'] == name].to_numpy()
    for ax, i in zip(row, rng.choice(members, min(8, len(members)), replace=False)):
        show(ax, i)
    for ax in row:
        ax.axis('off')
    row[0].text(-0.08, 0.5, f'{name}\n{len(members):,} train', transform=row[0].transAxes,
                ha='right', va='center', fontsize=11)
grid_title(fig, 'Random training images per class')
fig.savefig(RESULTS_OUT / '01_sample_grid.png', dpi=100)
plt.show()

## 5. Save

In [ ]:
df[['path', 'class_folder', 'class_name', 'label', 'split', 'group', 'md5']] \
    .sort_values(['label', 'path']).to_csv(DATA_OUT / 'splits.csv', index=False)
(DATA_OUT / 'classes.json').write_text(json.dumps(classes, indent=2))
table.to_csv(DATA_OUT / 'class_counts.csv')
info = {
    'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'dataset': 'Kaggle abdallahalidev/plantvillage-dataset, color folder, 10 tomato classes',
    'dataset_folder': str(COLOR_DIR),
    'paths_in_splits_csv': 'relative to the color folder',
    'seed': SEED,
    'split_fractions': SPLIT_FRACTIONS,
    'grouping_method': 'perceptual hash (DCT, 64 bits, 8 flips/rotations compared) '
                       '+ the same series and leaf number in the file name',
    'hash_threshold_bits': HASH_THRESHOLD,
    'images': N,
    'groups': int(n_groups),
    'images_sharing_a_group': int((group_size > 1).sum()),
    'numbered_leaves_in_file_names': int(leaf_id.nunique()),
    'photos_joined_by_file_name': int(len(same_leaf)),
    'of_those_missed_by_the_hash': int(missed.sum()),
    'groups_with_more_than_one_class': int(len(mixed)),
    'largest_gap_from_split_fractions_pct_points': round(float(off.max().max()), 1),
    'unreadable_images_left_out': bad,
    'split_sizes': {s: int((df['split'] == s).sum()) for s in SPLITS},
}
(DATA_OUT / 'split_info.json').write_text(json.dumps(info, indent=2))

for f in sorted(OUT_DIR.rglob('*')):
    if f.is_file() and f.parent.name in ('data', 'results'):
        print(f'{f.relative_to(OUT_DIR)}  ({f.stat().st_size / 1024:,.0f} KB)')

## Checklist before you accept the split

1. **Class counts:** 10 bars, about 18,160 images in total.
2. **Example pairs:** most pairs in the *same photo* rows are copies, and the *kept apart* rows are mostly different
   leaves. A few different leaves in the *same photo* rows are expected and harmless. Lower `HASH_THRESHOLD` by 2
   only if the *same photo* rows are mostly different leaves; raise it by 2 only if the *kept apart* rows just above
   the threshold are mostly copies.
3. **Largest groups:** these may be chains of different leaves; that is fine as long as the split table stays close
   to 70 / 15 / 15.
4. **Same leaf by file name:** each row is one leaf (or one plant on different days).
5. **Split table:** every class within about 1 percentage point of 70 / 15 / 15; the photos that share a group are
   spread about 70 / 15 / 15 too (not all in train); and all checks passed.
6. **Sample grid:** each row looks like its disease.

Then **Save Version → Save & Run All (Commit)** and download `data/` and `results/` into the project folder.
Next is notebook 02 (base models), which reads `splits.csv` from this notebook's output.